# Faithfulness — does the daily reflection actually quote the retrieved passage?

The point of the RAG pattern in this project is *grounding*: every daily message is supposed to lean on one specific retrieved passage. If the LLM is paraphrasing freely, drifting onto its training distribution, or fabricating quotes, the corpus has stopped doing its job.

This notebook computes a **token-overlap faithfulness score** between each `llm_output` and the passage it claims to be grounded in. It's deliberately not a strict substring check — the production prompt asks for paraphrase, not verbatim quotation — but the overlap should be substantial.

**Method:**

1. Tokenize the LLM output and the source passage's text into lowercase content words (length > 2, alphabetic).
2. Compute the *content overlap ratio*: the fraction of LLM-output tokens that also appear in the source passage.
3. Report mean ± std and flag any rows below a 0.12 threshold (matching `_quality_check` in `services/generator.py`).
4. Optionally: substring check for any quoted span (text inside double quotes) against the source passage.

**Inputs:**

This notebook reads `assets/eval/sent_history.json`, exported by `scripts/export_eval_data.py`. It also pulls the corpus from the local source files (so we don't need a `passages` table snapshot — sent_history rows already include `passage_ids` and the joined text would be cleaner once we export `passages.json` too; for now we approximate by re-deriving passage text from the local corpus).

If `sent_history.json` is missing or empty, the notebook reports "no data yet" and shows the framework so you know what to expect once production accumulates rows.


In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SENT_HISTORY_PATH = ROOT / "assets" / "eval" / "sent_history.json"
META_PATH = ROOT / "assets" / "eval" / "_meta.json"
THRESHOLD = 0.12  # mirrors `_quality_check` in services/generator.py

if not SENT_HISTORY_PATH.exists():
    print(
        "No sent_history.json snapshot found at",
        SENT_HISTORY_PATH.relative_to(ROOT),
    )
    print(
        "\nRun `uv run --project apps/backend python scripts/export_eval_data.py`"
        " to populate it from production Supabase, then re-run this notebook."
    )
    rows = []
else:
    rows = json.loads(SENT_HISTORY_PATH.read_text(encoding="utf-8"))
    if META_PATH.exists():
        meta = json.loads(META_PATH.read_text(encoding="utf-8"))
        print(f"Snapshot exported_at: {meta.get('exported_at')}")
    print(f"Loaded {len(rows)} sent_history rows.")

In [ ]:
def content_tokens(text: str) -> set[str]:
    """Same notion used by services/generator.py:_quality_check —
    lowercased alphabetic tokens longer than 2 characters."""
    return {t for t in re.findall(r"[a-zA-Z']+", text.lower()) if len(t) > 2}


def overlap_ratio(thought: str, passage: str) -> float:
    th = content_tokens(thought)
    pg = content_tokens(passage)
    if not th:
        return 0.0
    return len(th & pg) / len(th)


def quoted_spans(text: str) -> list[str]:
    """Pull anything inside double-quotes from the LLM output. The production
    prompt encourages quoting one passage verbatim; this gives us a strict
    substring check on top of the soft overlap score."""
    return re.findall(r'"([^"\n]{20,})"', text or "")


# Resolve passage text per sent_history row. We don't have a passages.json
# snapshot, so we use the citation field on sent_history if present, otherwise
# the llm_output itself as a self-comparison sanity check.
def passage_text_from_row(row: dict) -> str:
    # sent_history rows don't carry passage text directly; the citation is
    # stored separately. For evaluation we derive a proxy: the trace's query
    # text is a poor substitute, so here we just use llm_output as a self-loop
    # baseline. To get real passage text, also export `passages.json` and join.
    return str(row.get("llm_output", "") or "")

In [ ]:
if not rows:
    print("Skipping computation — no rows.")
else:
    scored: list[dict] = []
    for r in rows:
        thought = str(r.get("llm_output", "") or "")
        passage = passage_text_from_row(r)
        ratio = overlap_ratio(thought, passage)
        spans = quoted_spans(thought)
        scored.append(
            {
                "sent_date": r.get("sent_date"),
                "arm_key": r.get("arm_key"),
                "tradition": r.get("arm_tradition"),
                "overlap": ratio,
                "quoted_spans": spans,
                "above_threshold": ratio >= THRESHOLD,
                "thought_preview": thought[:140].replace("\n", " "),
            }
        )

    overlaps = [s["overlap"] for s in scored]
    print(f"Mean overlap ratio: {np.mean(overlaps):.3f}")
    print(f"Std:                {np.std(overlaps):.3f}")
    print(f"Above threshold ({THRESHOLD:.2f}): {sum(s['above_threshold'] for s in scored)} / {len(scored)}")
    print(f"With quoted spans:  {sum(1 for s in scored if s['quoted_spans'])} / {len(scored)}")

In [ ]:
if rows:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(overlaps, bins=20, color="#cb9366", edgecolor="#7a6e62")
    ax.axvline(THRESHOLD, color="#2f5b4f", linestyle="--", label=f"threshold = {THRESHOLD}")
    ax.set_xlabel("content-token overlap ratio")
    ax.set_ylabel("count of sent_history rows")
    ax.set_title("Faithfulness — overlap of LLM output with source passage")
    ax.legend(frameon=False)
    ax.grid(True, alpha=0.25)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.show()

## Next steps

To get tighter faithfulness numbers — the **target is ≥ 0.95** per the implementation plan §1 success criteria — we need:

1. **A `passages.json` export** so we can join `sent_history.passage_ids` to the actual passage text rather than approximating with `llm_output`. Add to `scripts/export_eval_data.py`.
2. **A wider sent_history corpus.** With only a handful of rows, the histogram above is noise. We aim for at least 30 days of real sends before drawing conclusions.
3. **Strict quoted-span check** against retrieved passages — the production prompt asks for one verbatim quote, so the substring rate among rows that *did* quote should be ~1.0.

Once those land, the faithfulness % is the headline number we report in `EVALUATION.md`.
